# 💧 Day 23 · Taste: the water Tashkent drinks, seen from space

**#30DayMapChallenge Uzbekistan 2026** · beginner tutorial, no experience needed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NodiraTillayeva/30DayMapChallengeUzbekistan/blob/main/2026/Day%2023%20Senses%20Taste/tutorial/day23_charvak_water.ipynb)

High in the mountains, 60 km from Tashkent, the **Pskem** and **Chatkal** rivers meet in the **Charvak reservoir**. The water that leaves it becomes the **Chirchik** river, and much of Tashkent's tap water comes from the Chirchik.

You can't taste water from space. But a satellite can see one thing that changes how water looks and tastes: **mud**. Muddy water reflects more light, especially red light, so it looks brown and bright. Clear water is dark.

In this notebook you will:
1. search the **Sentinel-2** satellite archive for clear days over Charvak,
2. compare the reservoir in **spring** and in **summer**,
3. find the water with a **water index** and measure its area,
4. measure **how muddy** every part of the water is,
5. find out **where the mud comes from**, and where it goes,
6. follow the reservoir from **spring to autumn**,
7. save a poster, and look at another lake.

No account needed: the data comes free from the **Microsoft Planetary Computer**.

Run the cells from top to bottom: click a cell, then press **Shift + Enter**.

## Step 0 · Get ready

The same tools as on Day 13: **pystac-client** searches the archive, **planetary-computer** gives free access, **odc-stac** loads the images.

We also write two small helpers:
- `find_days` lists the days when the satellite saw the **whole** box with few clouds,
- `load_day` loads one day and turns the numbers into **reflectance**: the share of sunlight the ground sends back, from 0 (black) to 1 (white).

In [ ]:
!pip -q install pystac-client planetary-computer odc-stac

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from shapely.geometry import box as make_box, shape
from shapely.ops import unary_union
from pyproj import Transformer
import pystac_client, planetary_computer, odc.stac

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

BOX = [69.94, 41.56, 70.14, 41.73]      # west, south, east, north: the Charvak reservoir

def find_days(box, start, end, max_clouds=10):
    items = catalog.search(collections=["sentinel-2-l2a"], bbox=box, datetime=f"{start}/{end}",
                           query={"eo:cloud_cover": {"lt": max_clouds}}).item_collection()
    area = make_box(*box)
    rows = []
    for day in sorted({i.datetime.date() for i in items}):
        today = [i for i in items if i.datetime.date() == day]
        seen = unary_union([shape(i.geometry) for i in today])
        if seen.contains(area):                       # the whole box, not just a part
            rows.append({"date": str(day), "clouds %": round(np.mean([i.properties["eo:cloud_cover"] for i in today]), 1)})
    return pd.DataFrame(rows)

def load_day(box, date):
    items = catalog.search(collections=["sentinel-2-l2a"], bbox=box, datetime=date).item_collection()
    lng = (box[0] + box[2]) / 2
    utm = f"EPSG:{32600 + int((lng + 180) // 6) + 1}"      # the UTM zone, in metres
    ds = odc.stac.load(items, bands=["B02", "B03", "B04", "B11", "SCL"], bbox=box,
                       crs=utm, resolution=20, groupby="solar_day").isel(time=0)
    for b in ["B02", "B03", "B04", "B11"]:
        ds[b] = (ds[b].where(ds[b] > 0) - 1000) / 10000    # numbers → reflectance (0–1)
    return ds

def true_colour(ds):
    rgb = np.dstack([ds[b].values for b in ["B04", "B03", "B02"]])
    return np.nan_to_num(rgb / 0.25).clip(0, 1) ** 0.8

print("Ready! ✅")

## Step 1 · Search for clear days

**Sentinel-2** is a pair of European satellites that photograph every place on Earth about every 5 days, with pixels of 10–20 m.

We ask for every picture of Charvak in **2025** with **less than 10% clouds** that shows the whole reservoir. How many clear days did each month have?

In [ ]:
days = find_days(BOX, "2025-01-01", "2025-12-31")
days["month"] = pd.to_datetime(days["date"]).dt.month

print(len(days), "clear days in 2025")
days.groupby("month").size().rename("clear days")

Winter has the fewest: clouds and snow. Summer has a clear day almost every time the satellite passes.

## Step 2 · Spring and summer

We load two days: **10 April** and **30 August 2025**, and show them in **true colour**: red, green and blue, like a photo.

In [ ]:
spring = load_day(BOX, "2025-04-10")
summer = load_day(BOX, "2025-08-30")

fig, axs = plt.subplots(1, 2, figsize=(16, 8))
for ax, ds, name in zip(axs, [spring, summer], ["10 April", "30 August"]):
    ax.imshow(true_colour(ds))
    ax.set_title(name)
    ax.axis("off")
plt.show()

Same reservoir, two different drinks. In April it's **small**, and its water is **milky brown**. In August it's **full** and **dark blue**.

## Step 3 · Find the water

To measure the water, we first have to find it. Water absorbs **short-wave infrared** light (band B11) almost completely, but reflects some **green** light (B03). The **water index** MNDWI compares the two:

MNDWI = (green − SWIR) / (green + SWIR)

Water is above 0, land is below. Snow is above 0 too, but it's very bright, so we also ask for green < 0.25. We leave out pixels that the satellite marked as clouds or shadows (`SCL`).

In [ ]:
def find_water(ds):
    green, swir = ds["B03"], ds["B11"]
    mndwi = (green - swir) / (green + swir)
    return (mndwi > 0.1) & (green < 0.25) & ~ds["SCL"].isin([3, 8, 9, 10])

def km2(water):
    return round(float(water.sum()) * 20 * 20 / 1_000_000, 1)    # each pixel is 20 m × 20 m

spring_water = find_water(spring)
summer_water = find_water(summer)
print("Water on 10 April: ", km2(spring_water), "km²")
print("Water on 30 August:", km2(summer_water), "km²")

fig, axs = plt.subplots(1, 2, figsize=(16, 8))
for ax, water, name in zip(axs, [spring_water, summer_water], ["10 April", "30 August"]):
    ax.imshow(water, cmap="Blues", vmin=0, vmax=1.5)
    ax.set_title(name)
    ax.axis("off")
plt.show()

In April the reservoir is only about **two thirds** of its summer size. Over winter, water is let out to make electricity and to keep the Chirchik flowing, and the reservoir gets low. Then in May and June, **melting snow** fills it again.

## Step 4 · How muddy is the water?

Clear water is dark: it reflects only 1–3% of red light. Mud floating in the water reflects much more red light. So we use the **red reflectance of the water**, in %, as a simple **mud index**:
- **under 3%**: clear water,
- **3–6%**: a bit cloudy,
- **over 6%**: muddy.

We show only the water; the land is grey.

In [ ]:
MUD = LinearSegmentedColormap.from_list("mud", ["#0b3c5d", "#2a9d8f", "#e9c46a", "#a0522d", "#5c3317"])

def mud_index(ds, water):
    return (ds["B04"] * 100).where(water)

spring_mud = mud_index(spring, spring_water)
summer_mud = mud_index(summer, summer_water)
print("Typical mud index on 10 April: ", round(float(spring_mud.median()), 1), "%")
print("Typical mud index on 30 August:", round(float(summer_mud.median()), 1), "%")

fig, axs = plt.subplots(1, 2, figsize=(16, 8))
for ax, mud, name in zip(axs, [spring_mud, summer_mud], ["10 April", "30 August"]):
    ax.set_facecolor("#d9d9d9")
    im = ax.imshow(mud, cmap=MUD, vmin=0, vmax=12)
    ax.set_title(name)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axs, shrink=0.6, label="mud index: red reflectance of the water (%)")
plt.show()

In April, the water is brown in the east and blue in the west. In August, it's blue everywhere.

## Step 5 · Where does the mud come from?

The two rivers come in from the east: the **Pskem** from the north-east, the **Chatkal** from the south-east. The dam is in the west. We measure the mud index at three places, as the average of a 100 m × 100 m square.

In [ ]:
places = {
    "Chatkal arm":     (41.615, 70.060),
    "Pskem arm":       (41.670, 70.040),
    "Near the dam":    (41.622, 69.985),
}

to_utm = Transformer.from_crs("EPSG:4326", spring.odc.crs, always_xy=True)

def sample(layer, lat, lng, half=50):
    x, y = to_utm.transform(lng, lat)
    return round(float(layer.sel(x=slice(x - half, x + half), y=slice(y + half, y - half)).mean()), 1)

pd.DataFrame([{"place": name,
               "10 April": sample(spring_mud, lat, lng),
               "30 August": sample(summer_mud, lat, lng)} for name, (lat, lng) in places.items()])

In April, the **Chatkal** brings the most mud, the **Pskem** less. But near the dam the water is already almost clear! The reservoir works like a **settling tank**: the water slows down, and the mud **sinks to the bottom** before it reaches the dam. That's good for the taste of the water downstream, and for the turbines.

Why April? Most likely because spring is the rainy season, and the first snow melts on the lower slopes: water runs over bare soil and carries it into the rivers. And with the reservoir low, the muddy rivers flow a long way over the shallow old river bed.

## Step 6 · From spring to autumn

If we took only the clearest day of each month, for April we would get **25 April**, and by then the mud had already settled. Mud comes in **bursts**, after rain, and is gone a couple of weeks later.

So we load **every clear day** from April to November 2025 at once (in winter and March, snow confuses the water index). To make it faster, we use bigger pixels, 40 m × 40 m, and the computer loads the days **in parallel** (`chunks={}` and `.compute()`). It takes 1–3 minutes.

In [ ]:
season = days[(days["month"] >= 4) & (days["month"] <= 11)]
found = catalog.search(collections=["sentinel-2-l2a"], bbox=BOX, datetime="2025-04-01/2025-11-30").item_collection()
items = [i for i in found if str(i.datetime.date()) in set(season["date"])]

cube = odc.stac.load(items, bands=["B03", "B04", "B11", "SCL"], bbox=BOX, crs=spring.odc.crs,
                     resolution=40, groupby="solar_day", chunks={})
for b in ["B03", "B04", "B11"]:
    cube[b] = (cube[b].where(cube[b] > 0) - 1000) / 10000
cube = cube.compute()

water = find_water(cube)
year = pd.DataFrame({
    "date": pd.to_datetime(cube.time.values).normalize(),
    "clouds %": (cube["SCL"].isin([3, 8, 9, 10]).mean(["x", "y"]) * 100).round(1).values,
    "water km²": (water.sum(["x", "y"]) * 40 * 40 / 1_000_000).round(1).values,
    "mud index %": mud_index(cube, water).median(["x", "y"]).round(1).values,
})
year = year[year["clouds %"] < 5]          # a cloud over the lake would hide some water
print(len(year), "days")
year.head(12)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(year["date"], year["water km²"], "o-", color="#1d6fa5")
ax1.set_ylabel("water (km²)")
ax2.plot(year["date"], year["mud index %"], "o-", color="#a0522d")
ax2.set_ylabel("mud index (%)")
for ax in (ax1, ax2):
    ax.grid(alpha=0.3)
plt.show()

Look at the two charts together:
- **Water:** about 23 km² on 10 April, then the snowmelt fills the reservoir to about **37 km²** in June. From July, more water goes out than comes in, and by October it's down to about 32 km².
- **Mud:** one big burst on **10 April** (about 8%), and two weeks later it's gone (1.4%). In June, the water is the clearest of the year (under 1%). In summer and autumn it stays low, 1.5–3%.

So if you want to *see* the taste of spring, you have to catch the right day.

## Step 7 · Save your poster

Spring and summer side by side. The land is the true-colour picture in grey, and the water is coloured by the mud index. Only tiny dates, no titles. It saves **charvak_water.png**.

In [ ]:
def grey_land(ds):
    return true_colour(ds).mean(axis=2) * 0.6

fig, axs = plt.subplots(1, 2, figsize=(16, 8), facecolor="#111111")
for ax, ds, mud, name in zip(axs, [spring, summer], [spring_mud, summer_mud], ["10 Apr 2025", "30 Aug 2025"]):
    ax.imshow(grey_land(ds), cmap="gray", vmin=0, vmax=1)
    ax.imshow(mud, cmap=MUD, vmin=0, vmax=12)
    ax.text(0.02, 0.02, name, transform=ax.transAxes, color="white", fontsize=10, alpha=0.8)
    ax.axis("off")
plt.subplots_adjust(wspace=0.02)
plt.savefig("charvak_water.png", dpi=200, bbox_inches="tight", facecolor="#111111")

try:
    from google.colab import files
    files.download("charvak_water.png")
except ImportError:
    print("Saved charvak_water.png")

## Step 8 · Another lake

Every reservoir has its own year. Pick a lake from the list (or add your own box), choose two months, and run the cell. It finds the clearest day in each month by itself.

In [ ]:
LAKES = {
    "Tuyabuguz (Tashkent Sea)": [69.27, 40.92, 69.38, 41.00],
    "Andijan reservoir":        [73.05, 40.73, 73.18, 40.82],
    "Kattakurgan reservoir":    [66.15, 39.72, 66.33, 39.86],
    "Talimarjan reservoir":     [65.50, 38.32, 65.62, 38.42],
}
lake = "Tuyabuguz (Tashkent Sea)"
months = ["2025-04", "2025-08"]

box = LAKES[lake]
fig, axs = plt.subplots(1, 2, figsize=(16, 8))
for ax, m in zip(axs, months):
    start = m + "-01"
    end = (pd.Timestamp(start) + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")
    clear = find_days(box, start, end)
    if clear.empty:
        ax.set_title(m + ": no clear day 😶"); ax.axis("off"); continue
    date = clear.sort_values("clouds %")["date"].iloc[0]
    ds = load_day(box, date)
    water = find_water(ds)
    mud = mud_index(ds, water)
    ax.imshow(grey_land(ds), cmap="gray", vmin=0, vmax=1)
    ax.imshow(mud, cmap=MUD, vmin=0, vmax=12)
    ax.set_title(f"{date}: {km2(water)} km², mud index {round(float(mud.median()), 1)}%")
    ax.axis("off")
plt.show()

## 🎉 Done!

You searched a satellite archive, found the water, measured its area and how muddy it is, and followed Charvak from spring to autumn.

Share your map with **#30DayMapChallenge** and **#30DayMapChallengeUzbekistan**, and please credit:
> Contains modified Copernicus Sentinel-2 data (2025), accessed through the Microsoft Planetary Computer.